# Natural Language Processing Lab - Introduction
## Task 2: NLP Preprocessing, Linguistic Tagging & Feature Extraction Pipeline

**Objective**:
Design and evaluate an end-to-end NLP preprocessing pipeline on noisy, informal web text. The pipeline performs:
1. Lexical cleaning (slang conversion, contraction expansion, URL/symbol removal, character repetition reduction)
2. Tokenization & vocabulary frequency analysis
3. Morphological analysis (Porter Stemming vs WordNet Lemmatization)
4. Named Entity Recognition (NER via spaCy)
5. Part-of-Speech (POS) Tagging (NLTK)
6. Term Frequency-Inverse Document Frequency (TF-IDF) feature matrix computation

In [ ]:
import re
import nltk
import spacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

# Download required NLTK resources
for resource in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

# Initialize SpaCy pipeline
nlp_engine = spacy.load("en_core_web_sm")
print("Environment and NLP packages initialized successfully.")

### 1. Raw Text Corpora Collection
Five noisy, informal paragraphs representing social media comments, chat streams, and forum discussions.

In [ ]:
raw_documents = [
    """OMG!! You won't BELIEVE what happened today at the AI conference in San Francisco! 
    Dr. Smith presented this aWesoMe new model for leanring representations from raw text. 
    Check out the paper here: https://arxiv.org/abs/2301.99999 -- it is seriously mindblowing! 
    Can't wait to try it out tomorrow morning... #AI #MachineLearning #NLP2026 @StanfordNLP <3""",

    """Hey guys, quick question: what is the BEST way to handle out-of-vocabulary words in modern NLP? 
    I'm working on a chatbot for customer service and it keeps failing on slang words like 'lit', 'dope', and 'fr'. 
    Should I use Byte-Pair Encoding (BPE) or WordPiece? Any tips/tricks would be greatly appreciated! 
    Email me at alex_nlp_dev@gmail.com or DM me on Twitter @alex_codes.""",

    """The quick brown fox jumps over the lazy dog. But in modern NLP, we care more about transformers like BERT, 
    GPT-4, and RoBERTa developed by OpenAI and Google in California. Natural Language Processing has evolved from 
    simple rule-based systems to massive deep learning models with billions of parameters! 
    Cost of training is $$$$$ though... :( Visit https://huggingface.co for pretrained models.""",

    """Breaking: Apple Inc. announced a new $500M investment in their AI research facility in Seattle, Washington. 
    CEO Tim Cook stated that the facility will focus on on-device NLP, speech recognition, and privacy-preserving ML. 
    The initiative will create over 1,200 new engineering jobs by December 2026. 
    Read full coverage at www.techcrunch.com/apple-ai-investment-2026/!""",

    """heyyyyy, did u read that crazyyyyyyy article on quantum stuff & ai?? 
    like for realll... it wuz crazzzzy detailed but sooooo complicateddd!!! 
    i cud barely undrstnd half of it. the diagrams were fire though!!! 
    i found it here – www.quantmgeekzz.net//// … plz plz plz read n 
    tell me what u thinkk #toomuchinfo #brainhurts @@@!!!"""
]

print(f"Total documents loaded: {len(raw_documents)}")
for idx, doc in enumerate(raw_documents, 1):
    print(f"\n--- Raw Document {idx} Preview (Length: {len(doc)} chars) ---")
    print(doc.strip()[:140] + "...")

### 2. Text Normalization Pipeline
A robust cleaning function standardizing abbreviations, contractions, URLs, symbols, and character floods.

In [ ]:
# Mapping dictionary for contractions and common informal slang
CONTRACTION_SLANG_MAP = {
    "can't": "cannot",
    "won't": "will not",
    "i'm": "i am",
    "i've": "i have",
    "it's": "it is",
    "i'll": "i will",
    "i'd": "i would",
    "you're": "you are",
    "u're": "you are",
    "waz": "was",
    "wuz": "was",
    "cud": "could",
    "c'mon": "come on",
    "<3": "love",
    "omg": "oh my god",
    "plz": "please",
    "undrstnd": "understand",
    "fr": "for real",
    "realll": "real",
    "leanring": "learning",
    "awesome": "awesome",
    "awesom": "awesome",
    "thinkk": "think"
}

def clean_noisy_text(text: str) -> str:
    # 1. Lowercase normalization
    normalized = text.lower()
    
    # 2. Expand contractions & standardized slang
    for slang, standard in CONTRACTION_SLANG_MAP.items():
        normalized = re.sub(rf"\b{re.escape(slang)}\b", standard, normalized)
        
    # 3. Strip URLs and domain artifacts
    normalized = re.sub(r"https?://\S+|www\.\S+", " ", normalized)
    
    # 4. Strip email addresses
    normalized = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", " ", normalized)
    
    # 5. Compress elongated characters (e.g., 'crazyyyyyyy' -> 'crazy')
    normalized = re.sub(r"(.)\1{2,}", r"\1\1", normalized)
    
    # 6. Filter out symbols and non-alphanumeric noise
    normalized = re.sub(r"[^a-z0-9\s]", " ", normalized)
    
    # 7. Normalize consecutive whitespace
    cleaned = re.sub(r"\s+", " ", normalized).strip()
    return cleaned

cleaned_documents = [clean_noisy_text(doc) for doc in raw_documents]

for idx, (raw, cleaned) in enumerate(zip(raw_documents, cleaned_documents), 1):
    print(f"\n{'='*70}\n[Cleaned Document {idx}]\n{'='*70}\n{cleaned}")

### 3. Tokenization & Word Frequency Distribution
Tokenizing cleaned documents and analyzing lexical frequencies.

In [ ]:
# Tokenize all cleaned documents
tokenized_corpus = [word_tokenize(doc) for doc in cleaned_documents]
aggregated_tokens = [tok for doc_tokens in tokenized_corpus for tok in doc_tokens]

token_counts = Counter(aggregated_tokens)
top_20_tokens = token_counts.most_common(20)

print(f"Total tokens across corpus: {len(aggregated_tokens)}")
print(f"Unique vocabulary size    : {len(token_counts)}")
print("\nTop 20 Most Frequent Tokens:")
for rank, (tok, freq) in enumerate(top_20_tokens, 1):
    print(f"{rank:02d}. {tok:<15} : {freq}")

In [ ]:
# Plotting Top 20 Most Frequent Words
plt.figure(figsize=(12, 5), dpi=100)
words, freqs = zip(*top_20_tokens)

sns.barplot(x=list(words), y=list(freqs), palette="viridis")
plt.title("Top 20 Most Frequent Words in Cleaned Corpus", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Tokens", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

### 4. Morphological Analysis: Stemming vs Lemmatization
Comparing rule-based suffix stripping (`PorterStemmer`) with lexicon-based morphological reduction (`WordNetLemmatizer`).

In [ ]:
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

benchmark_words = [
    "learning", "articles", "changing", "amazing", "spelling",
    "watched", "stumbled", "blogs", "symbols", "crazy",
    "detailed", "spaces", "weird", "complicated", "thoughts"
]

comparison_rows = []
for word in benchmark_words:
    stem_result = porter.stem(word)
    lemma_result = lemmatizer.lemmatize(word)
    divergence = "Yes" if stem_result != lemma_result else "No"
    
    comparison_rows.append({
        "Original Word": word,
        "Porter Stemmer": stem_result,
        "WordNet Lemmatizer": lemma_result,
        "Divergence": divergence
    })

comparison_df = pd.DataFrame(comparison_rows)
print("Stemming vs. Lemmatization Comparison Table:")
display(comparison_df)

### 5. Information Extraction: Named Entity Recognition (NER)
Extracting real-world named entities across the corpus using spaCy.

In [ ]:
# Concatenate cleaned documents for collective NER
full_cleaned_corpus = " ".join(cleaned_documents)
spacy_doc = nlp_engine(full_cleaned_corpus)

ner_records = []
for entity in spacy_doc.ents:
    ner_records.append({
        "Entity Text": entity.text,
        "Label": entity.label_,
        "Description": spacy.explain(entity.label_)
    })

ner_df = pd.DataFrame(ner_records).drop_duplicates().reset_index(drop=True)
print("Extracted Named Entities:")
display(ner_df)

### 6. Grammatical Tagging: Part-of-Speech (POS)
Generating POS tags for the tokenized sentences using NLTK's Perceptron Tagger.

In [ ]:
for idx, doc_tokens in enumerate(tokenized_corpus, 1):
    pos_tags = pos_tag(doc_tokens)
    print(f"\n--- POS Tagging: Document {idx} (First 10 Tokens) ---")
    print(pos_tags[:10])

### 7. Vector Space Modeling: TF-IDF Representation
Transforming the text collection into a TF-IDF term matrix.

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_documents)
feature_vocabulary = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=[f"Doc_{i}" for i in range(1, len(cleaned_documents) + 1)],
    columns=feature_vocabulary
)

print(f"TF-IDF Matrix Dimension: {tfidf_df.shape[0]} Documents x {tfidf_df.shape[1]} Features")
display(tfidf_df.iloc[:, :15].round(3))